# 1. Imports and global settings

In [ ]:
import os

# Control warning logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [2]:
import os
import random
import numpy as np
import torch
from torchvision import datasets, transforms
from tqdm import tqdm
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.get_logger().setLevel("ERROR")

from scripts.pruning.device import configure_device
DEVICE = configure_device("gpu:0")  # Change to cpu for CPU execution.


In [3]:
# Reproducibility seed
# Using a fixed seed helps make results more repeatable
SEED = 42

def fix_random_seed(seed=SEED):
    """Fix random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    tf.random.set_seed(seed)

fix_random_seed(SEED)

print("Setup complete")

# 2. Data loading

In [ ]:
from scripts.pruning.data import load_data_pt


In [5]:
# File paths for train, validation, and test.
train_path = os.path.join("./data/train.pt")
val_path = os.path.join("./data/val.pt")
test_path = os.path.join("./data/test.pt")

# Load files if they exist.
if all(os.path.exists(p) for p in [train_path, val_path, test_path]):
    x_train, y_train, meta_train = load_data_pt(train_path, normalized=False)
    x_val, y_val, meta_val = load_data_pt(val_path, normalized=False)
    x_test, y_test, meta_test = load_data_pt(test_path, normalized=False)
    
    print("Data loaded successfully.")
    print("Train:", x_train.shape, y_train.shape)
    print("Val:  ", x_val.shape, y_val.shape)
    print("Test: ", x_test.shape, y_test.shape)
    print("You can now proceed to model training and evaluation.")
else:
    print("Dataset files not found yet.")

# 3. Visualization

In [6]:
def multihot_to_digits(multihot):
    """
    Convert a multi-hot vector into a list of digit labels.

    Example:
        [0, 1, 0, 0, 1, 0, 0, 0, 0, 1]
        becomes
        [1, 4, 9]
    """
    return np.where(multihot >= 0.5)[0].tolist()


def visualize_one_sample(images, labels, meta, idx=0):
    """Visualize one generated image and its labels."""
    img = images[idx].squeeze()

    digits_present = multihot_to_digits(labels[idx])
    center_label = meta["center_labels"][idx]
    all_labels = meta["all_digit_labels"][idx]
    n_digits = meta["num_digits"][idx]

    # Remove padding values.
    valid_all_labels = all_labels[all_labels >= 0].tolist()

    plt.figure(figsize=(4, 4))
    plt.imshow(img, cmap="gray")
    plt.title(
        f"multi-label={digits_present}\n"
        f"center={center_label}, all={valid_all_labels}, n={n_digits}"
    )
    plt.axis("off")
    plt.show()


def visualize_grid(images, labels, meta, num_samples=16):
    """Visualize a grid of random samples."""
    rows = int(np.sqrt(num_samples))
    cols = int(np.ceil(num_samples / rows))

    plt.figure(figsize=(10, 10))

    for i in range(num_samples):
        idx = random.randint(0, len(images) - 1)

        img = images[idx].squeeze()
        digits_present = multihot_to_digits(labels[idx])
        center_label = meta["center_labels"][idx]

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img, cmap="gray")
        plt.title(f"y={digits_present}\nc={center_label}", fontsize=8)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


# Visualize samples if the dataset has been loaded.
if "x_train" in globals():
    visualize_one_sample(x_train, y_train, meta_train, idx=0)
    visualize_grid(x_train, y_train, meta_train, num_samples=16)

# 3. Define exact match accuracy metrics:

In [7]:
from scripts.pruning.model import exact_match_accuracy


# 4. Build the ResNet-18 model

In [ ]:
from scripts.pruning.model import build_resnet18_model


In [9]:
# Save untouched weights and BatchNorm state before compiling.
from pathlib import Path
from scripts.pruning.storage import prepare_output, write_json
from scripts.pruning.config import TrainingConfig, BenchmarkConfig
from scripts.pruning.training import save_initialization, train
from scripts.pruning.model import compile_model

MODEL_DIR = Path("models/dense")
RESULT_PATH = Path("results/baseline.json")
prepare_output(RESULT_PATH, MODEL_DIR)
INPUT_SHAPE = (64, 64, 3)
model = build_resnet18_model(input_shape=INPUT_SHAPE, dropout_seed=SEED)
INITIAL_PATH = save_initialization(model, MODEL_DIR)


In [ ]:
TRAINING = TrainingConfig(device=DEVICE)
compile_model(model, TRAINING)


In [11]:
# Display the model architecture.
model.summary()

# 5. Train the model

In [ ]:
BATCH_SIZE = TRAINING.batch_size
EPOCHS = TRAINING.epochs


In [ ]:
# Shared training reloads the best validation epoch after EMA finalization.
model, training_history = train(
    model, (x_train, y_train), (x_val, y_val), TRAINING, MODEL_DIR, seed=SEED,
)
history = training_history["history"]


In [14]:
# Plot training history if available.

if "history" in globals():
    plt.figure(figsize=(12, 4))

    # Loss curve.
    plt.subplot(1, 2, 1)
    plt.plot(history["loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and validation loss")
    plt.legend()

    # Exact-match accuracy curve.
    plt.subplot(1, 2, 2)
    plt.plot(history["exact_match_accuracy"], label="train_exact_match")
    plt.plot(history["val_exact_match_accuracy"], label="val_exact_match")
    plt.xlabel("Epoch")
    plt.ylabel("Exact-match accuracy")
    plt.title("Training and validation binary accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("No training history found.")

# 6. Evaluate the model

The test set is used only after training.

Important:

- `binary_accuracy` measures label-wise correctness.
- `precision` measures whether predicted labels are correct.
- `recall` measures whether true labels are found.
- `exact_match_accuracy` checks whether the whole label set is correct for each image.

For this task, precision, recall, and exact-match accuracy are usually more informative than binary accuracy.

In [ ]:
from scripts.pruning.benchmark import evaluate_and_benchmark
from scripts.pruning.reference import baseline_result

BENCHMARK = BenchmarkConfig(
    batch_size=BATCH_SIZE,
    device=TRAINING.device,
)
metrics = evaluate_and_benchmark(model, (x_test, y_test), MODEL_DIR / "model.keras", BENCHMARK)
results = baseline_result(MODEL_DIR, TRAINING, BENCHMARK, SEED, training_history, metrics)
results["status"] = "completed"
write_json(RESULT_PATH, results)
print(metrics)
